# 04 — Train: Temporal Fusion Transformer

A strict, notebook-local TFT experiment for station `207241-at`. The train and sealed-test feature artifacts are kept physically independent: a test encoder window never contains a train observation.

This is the only stage-4 candidate that forecasts a distribution rather than a point. It is trained with quantile loss and emits p10, p50 and p90 at every one of the 24 horizons, so it is scored on calibration as well as accuracy — and it is also the only one that can report which predictors it used.

**Inputs:** train-derived and sealed-test Stage-3 feature artifacts
**Outputs:** in-notebook quantile forecasts, accuracy and calibration tables, and interpretation figures only — no checkpoint, no saved model, nothing written to disk

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins every knob of the run. Like the other stage-4 notebooks this fixes exactly one candidate — no tuning, no validation split, no checkpoints — but unlike the others it produces a *probabilistic* forecast rather than a single number per horizon.

**What the imports provide**

- `TemporalFusionTransformer` — an attention-based forecaster that combines a recurrent encoder, variable-selection networks that learn which inputs matter, and interpretable multi-head attention over the encoder window.
- `TimeSeriesDataSet` — pytorch-forecasting's sample builder. It slices a long frame into encoder/decoder windows, tracks which variables are known in advance, and fits the normalisers.
- `TorchNormalizer` — target scaling, fitted on train and reused for test.
- `QuantileLoss` — the pinball loss that lets one model emit several quantiles at once.
- `Trainer`, `lightning.pytorch` — the training loop and its seeding.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `SEED` | `42` | Seeds Lightning, Python, NumPy and Torch (with `workers=True`, also the dataloader workers) so the run reproduces. Note `deterministic_kernels` is reported as `False` in the configuration table: MPS kernels are not bit-reproducible, so the run repeats closely, not exactly. |
| `ENCODER_LENGTH` | `168` | Hours of history fed to the encoder — one week. Longer windows give attention more to work with and cost memory quadratically in the attention step. |
| `PREDICTION_LENGTH` | `24` | Decoder steps, i.e. the forecast horizon. The guard below asserts it equals the number of Stage-3 target columns, so the TFT and the other candidates forecast exactly the same span. |
| `QUANTILES` | `(0.10, 0.50, 0.90)` | The quantiles the model is trained to emit at every horizon. `0.50` (the median) is the point forecast used for MAE/RMSE; `0.10` and `0.90` bound a nominal 80% prediction interval. This is the only stage-4 candidate that says anything about its own uncertainty. |
| `BATCH_SIZE` | `64` | Samples per gradient step, used for both dataloaders. |
| `MAX_EPOCHS` | `10` | Full passes over the training samples. With no validation loop and no early stopping this is a fixed budget, and it is small — the TFT is by far the most expensive candidate per epoch. |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 Parquets are read from. Nothing is written back; not even a checkpoint. |
| `CALENDAR_COLUMNS` | 6 names | The sine/cosine Fourier pairs for hour-of-day, day-of-week and day-of-year. These are the only predictors that are genuinely *known in advance*, so they are the only ones the decoder is allowed to see for future steps. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract. |
| `ENCODER_ONLY_REALS` | the other 47 | Water level, the imputation flag, weather, lags and rolling statistics. None of them can be known for a future hour at issue time, so they are restricted to the encoder. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | Used only as a horizon check here. The TFT forecasts the `water_level` series over 24 decoder steps directly rather than reading these columns. |

**The guards, and why they abort rather than adapt**

- The station is pinned to `207241-at`; a different `TARGET_STATION_ID` raises rather than silently producing an unrelated experiment.
- `PREDICTION_LENGTH` must match the Stage-3 horizon.
- MPS must be available. The CPU fallback is deliberately disabled: on CPU this configuration takes long enough that a run started by accident would look like a hang, and the experiment being reported would no longer be the one that was configured.

The final table is the run's record of the fixed hyperparameters, including the model-shape values (`hidden_size`, `lstm_layers`, `attention_heads`, `hidden_continuous_size`, `dropout`, `learning_rate`, `gradient_clip_val`) that are passed literally in the training cell further down and documented there.

In [ ]:
import random
from pathlib import Path

import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from lightning.pytorch import Trainer
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import TorchNormalizer
from pytorch_forecasting.metrics import QuantileLoss

from src.config import TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

SEED = 42
ENCODER_LENGTH = 168
PREDICTION_LENGTH = 24
QUANTILES = (0.10, 0.50, 0.90)
BATCH_SIZE = 64
MAX_EPOCHS = 10
PROCESSED_DIR = Path("data/processed")

CALENDAR_COLUMNS = [
    "utc_hour_sin",
    "utc_hour_cos",
    "utc_day_of_week_sin",
    "utc_day_of_week_cos",
    "utc_day_of_year_sin",
    "utc_day_of_year_cos",
]
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())
ENCODER_ONLY_REALS = [
    column for column in FEATURE_COLUMNS if column not in CALENDAR_COLUMNS
]

if TARGET_STATION_ID != "207241-at":
    raise ValueError(
        f"This MVP is fixed to station 207241-at, got {TARGET_STATION_ID!r}"
    )
if PREDICTION_LENGTH != len(TARGET_COLUMNS):
    raise ValueError("TFT prediction length must match the Stage-3 target horizon")
if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS is required for this experiment and is not available; CPU fallback is disabled"
    )

pl.seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

configuration = pd.DataFrame(
    [
        {
            "station_id": TARGET_STATION_ID,
            "device": "mps",
            "seed": SEED,
            "encoder_hours": ENCODER_LENGTH,
            "forecast_hours": PREDICTION_LENGTH,
            "quantiles": QUANTILES,
            "batch_size": BATCH_SIZE,
            "epochs": MAX_EPOCHS,
            "hidden_size": 64,
            "lstm_layers": 2,
            "attention_heads": 4,
            "hidden_continuous_size": 32,
            "dropout": 0.1,
            "learning_rate": 0.001,
            "gradient_clip_val": 0.1,
            "target_normalization": "standard, fit on train station only",
            "deterministic_kernels": False,
            "known_decoder_reals": len(CALENDAR_COLUMNS),
            "encoder_only_reals": len(ENCODER_ONLY_REALS),
        }
    ]
)
display(configuration)

## Input contract and leakage boundary

Only the Stage-3 train and sealed-test Parquets are read, and each is processed on its own — so a test encoder window can never contain a training observation.

**`read_feature_artifact(path, *, station_id, artifact_name)`** rejects an artifact unless every required column is present, it is non-empty, it contains exactly one station, its timestamps are timezone-aware and duplicate-free, and every `target_valid` row carries a complete target vector. It also casts the Boolean `imputed` flag to `float`, because TFT treats it as a continuous input and cannot take a Boolean — a conversion made on the notebook-local copy only, never on disk.

**`build_complete_segments(frame, *, artifact_name)`** does the work that makes fixed-length windows safe:

- keeps only rows where all 53 predictors are present;
- starts a new **segment** wherever completeness breaks or the hourly timeline jumps, so `segment_id` labels a run of genuinely contiguous, fully populated hours. Segments are partition-local identifiers rather than features — they are what lets the sealed test reuse the train-fitted categorical encoders without the two ever being joined;
- computes `time_idx`, an integer hour counter since 1970-01-01 UTC, because `TimeSeriesDataSet` needs a monotone integer clock rather than timestamps;
- casts the predictors to `float32`, since MPS has no `float64` support.

Finally the two splits are asserted to be chronological and non-overlapping — the last train timestamp must precede the first test timestamp.

In [ ]:
def read_feature_artifact(
    path: Path, *, station_id: str, artifact_name: str
) -> pd.DataFrame:
    """Load one Stage-3 artifact and enforce the TFT input contract."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing feature artifact for {station_id}: {path}")
    frame = pd.read_parquet(path).copy()
    required = {
        "timestamp",
        "station_id",
        "target_valid",
        *FEATURE_COLUMNS,
        *TARGET_COLUMNS,
    }
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")
    if frame["station_id"].dropna().unique().tolist() != [station_id]:
        raise ValueError(
            f"{station_id} {artifact_name} artifact contains another station"
        )
    frame = frame.sort_values("timestamp").reset_index(drop=True)
    if frame["timestamp"].duplicated().any():
        raise ValueError(
            f"{station_id} {artifact_name} artifact has duplicate timestamps"
        )
    if not pd.api.types.is_datetime64tz_dtype(frame["timestamp"]):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be timezone-aware"
        )
    target_rows = frame["target_valid"].eq(True)
    if frame.loc[target_rows, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} has incomplete target vectors in target-valid rows"
        )
    frame["imputed"] = frame["imputed"].astype(float)
    return frame


def build_complete_segments(frame: pd.DataFrame, *, artifact_name: str) -> pd.DataFrame:
    """Keep only contiguous rows with every Stage-3 predictor present."""
    complete = frame[FEATURE_COLUMNS].notna().all(axis=1)
    contiguous = frame["timestamp"].diff().eq(pd.Timedelta(hours=1))
    starts_segment = complete & (~complete.shift(fill_value=False) | ~contiguous)
    segment_number = starts_segment.cumsum()
    result = frame.loc[complete].copy()
    # Segment labels are partition-local, stable identifiers rather than features.
    # This lets the sealed test reuse fitted categorical encoders without joining it to train.
    result["segment_id"] = "segment_" + segment_number.loc[complete].astype(
        str
    ).str.zfill(4)
    result["time_idx"] = (
        (result["timestamp"] - pd.Timestamp("1970-01-01", tz="UTC"))
        / pd.Timedelta(hours=1)
    ).astype(int)
    # MPS does not support float64 tensors; this copy is notebook-local.
    result[FEATURE_COLUMNS] = result[FEATURE_COLUMNS].astype(np.float32)
    if result.empty:
        raise ValueError(f"{artifact_name} artifact has no complete feature rows")
    return result.reset_index(drop=True)


station_id = TARGET_STATION_ID
train_features = read_feature_artifact(
    PROCESSED_DIR / f"{station_id}_train_features.parquet",
    station_id=station_id,
    artifact_name="train",
)
test_features = read_feature_artifact(
    PROCESSED_DIR / f"{station_id}_test_features.parquet",
    station_id=station_id,
    artifact_name="test",
)
train_data = build_complete_segments(train_features, artifact_name="train")
test_data = build_complete_segments(test_features, artifact_name="test")

if train_data["timestamp"].max() >= test_data["timestamp"].min():
    raise ValueError("Train and sealed test artifacts overlap or are not chronological")

## Fixed-length, complete sequences

**`make_dataset(data, *, template=None)`** builds the samples. Called without a `template` it constructs the training dataset from scratch; called with one it reuses that dataset's fitted encoders and normalisers for the test split.

| Argument | Value | What it does |
| --- | --- | --- |
| `time_idx` | `"time_idx"` | The integer hour clock built above; the library orders and slices on it. |
| `target` | `"water_level"` | The TFT forecasts the series itself over 24 decoder steps, instead of reading Stage-3's 24 pre-computed target columns. |
| `group_ids` | `["segment_id"]` | One independent series per contiguous complete segment, which is what prevents any window from spanning a gap. |
| `min_encoder_length` / `max_encoder_length` | `168` / `168` | Both equal, so every sample has exactly one week of history — no variable-length encoders. |
| `min_prediction_length` / `max_prediction_length` | `24` / `24` | Likewise fixed at the full horizon. |
| `static_categoricals` | `["station_id"]` | Constant per series. It carries no information with a single station, but keeping it means the fitted categorical encoder transfers unchanged to the test set. |
| `time_varying_known_reals` | the 6 calendar columns | Available to the **decoder** at future steps, because clock values are computable for any timestamp. |
| `time_varying_unknown_reals` | the other 47 predictors | Encoder-only: the model may use their history but never their future. |
| `target_normalizer` | `TorchNormalizer(method="standard")` | Z-scores the target. Fitted on train and reused for test through `from_dataset`, and inverted automatically so predictions come back in water-level units. |
| `allow_missing_timesteps` | `False` | Refuses to stitch across a missing hour. Combined with the segmentation above, this guarantees every sample is physically contiguous. |
| `randomize_length` | `False` | Deterministic sample construction — no random window shortening. |
| `stop_randomization=True`, `predict=False` (test only) | | Disables the training-time randomisation for the test split while keeping *all* valid windows rather than only the last one per segment. |

**`filter_target_valid_sequences(dataset, data, *, artifact_name)`** then narrows the candidates to the scoring cohort. A sample survives only if the hour immediately before its first decoder step — its issue time — was marked `target_valid` by Stage 3, i.e. all 24 following hours were genuinely observed. This is the same cohort rule the other stage-4 notebooks apply, expressed against decoder indices. It raises if a split has no surviving sample.

The displayed table shows the attrition at each stage: complete rows, candidate windows, and finally scored sequences. Short or fragmented segments are intentionally absent from the strict cohort.

In [ ]:
def make_dataset(
    data: pd.DataFrame, *, template: TimeSeriesDataSet | None = None
) -> TimeSeriesDataSet:
    """Construct fixed-length TFT samples without allowing missing timesteps."""
    if template is not None:
        return TimeSeriesDataSet.from_dataset(
            template, data, stop_randomization=True, predict=False
        )
    return TimeSeriesDataSet(
        data,
        time_idx="time_idx",
        target="water_level",
        group_ids=["segment_id"],
        min_encoder_length=ENCODER_LENGTH,
        max_encoder_length=ENCODER_LENGTH,
        min_prediction_length=PREDICTION_LENGTH,
        max_prediction_length=PREDICTION_LENGTH,
        static_categoricals=["station_id"],
        time_varying_known_reals=CALENDAR_COLUMNS,
        time_varying_unknown_reals=ENCODER_ONLY_REALS,
        target_normalizer=TorchNormalizer(method="standard"),
        allow_missing_timesteps=False,
        randomize_length=False,
    )


def filter_target_valid_sequences(
    dataset: TimeSeriesDataSet, data: pd.DataFrame, *, artifact_name: str
) -> TimeSeriesDataSet:
    """Keep samples whose decoder starts immediately after a valid issue time."""
    eligibility = data.set_index(["segment_id", "time_idx"])["target_valid"]

    def issue_is_eligible(index: pd.DataFrame) -> pd.Series:
        issue_keys = list(
            zip(index["segment_id"], index["time_idx_first_prediction"] - 1)
        )
        return pd.Series(
            [bool(eligibility.get(key, False)) for key in issue_keys], index=index.index
        )

    filtered = dataset.filter(issue_is_eligible, copy=True)
    if len(filtered) == 0:
        raise ValueError(
            f"{artifact_name} artifact has no fixed {ENCODER_LENGTH}-to-{PREDICTION_LENGTH} target-valid sequences"
        )
    return filtered


train_candidates = make_dataset(train_data)
test_candidates = make_dataset(test_data, template=train_candidates)
train_dataset = filter_target_valid_sequences(
    train_candidates, train_data, artifact_name="train"
)
test_dataset = filter_target_valid_sequences(
    test_candidates, test_data, artifact_name="test"
)

candidate_sequence_counts = pd.DataFrame(
    [
        {
            "artifact": "train",
            "complete_rows": len(train_data),
            "candidate_sequences": len(train_candidates),
            "scored_sequences": len(train_dataset),
        },
        {
            "artifact": "test",
            "complete_rows": len(test_data),
            "candidate_sequences": len(test_candidates),
            "scored_sequences": len(test_dataset),
        },
    ]
)
display(candidate_sequence_counts)

## TFT training

Variable routing was fixed in the previous cell: the six calendar signals are known to the decoder, while water level, the imputation flag, weather, lags and rolling features are encoder-only. Feature and target normalisers were fitted on train and are reused unchanged for test.

**Model parameters** (`TemporalFusionTransformer.from_dataset`, which reads the variable roles and normalisers straight off `train_dataset`):

| Parameter | Value | What it does |
| --- | --- | --- |
| `hidden_size` | `64` | Width of the main hidden state throughout the network. The single biggest capacity and cost knob. |
| `lstm_layers` | `2` | Depth of the recurrent encoder/decoder that runs beneath the attention layer. |
| `attention_head_size` | `4` | Number of interpretable attention heads over the encoder window. More heads let the model attend to several distinct points in the past week at once. |
| `hidden_continuous_size` | `32` | Width of the per-variable network applied to each continuous input before variable selection. Kept below `hidden_size`, which is the usual convention — with 53 inputs this layer is a large share of the parameter count. |
| `dropout` | `0.1` | Fraction of units dropped during training. The only regulariser here, since there is neither early stopping nor weight decay. |
| `learning_rate` | `0.001` | Adam's step size, held constant. |
| `loss` | `QuantileLoss(quantiles=[0.10, 0.50, 0.90])` | Pinball loss summed over the three quantiles. It penalises under- and over-prediction asymmetrically per quantile, which is what makes one model produce a calibrated spread instead of three separate fits. |
| `output_size` | `3` | Values emitted per horizon — one per quantile. It must match the loss's quantile count. |
| `optimizer` | `"Adam"` | Plain Adam rather than pytorch-forecasting's Ranger default, so the run is easy to reason about and reproduce. |
| `reduce_on_plateau_patience` | `0` | Leaves no patience for the plateau learning-rate scheduler, and with no validation dataloader there is nothing for it to monitor — so the learning rate is effectively constant. |
| `log_interval` | `-1` | Disables per-batch logging of predictions and interpretation plots. |
| `logging_metrics` | `[]` | No extra metrics are tracked during training; accuracy is measured once, below, on the sealed test set. |

**Trainer parameters:**

| Parameter | Value | What it does |
| --- | --- | --- |
| `accelerator` / `devices` | `"mps"` / `1` | Train on the single Apple GPU, matching the hard requirement asserted in Setup. |
| `max_epochs` | `MAX_EPOCHS` (10) | The full training budget. |
| `gradient_clip_val` | `0.1` | Rescales gradients whose norm exceeds 0.1. Deep recurrent-plus-attention models are prone to exploding gradients; this is a cheap stabiliser and pytorch-forecasting's recommended value for TFT. |
| `logger` | `False` | No TensorBoard or CSV run directory is created. |
| `enable_checkpointing` | `False` | No weights are written to disk — consistent with the stage-4 rule that a run produces numbers, not artifacts. |
| `enable_model_summary` | `False` | Suppresses the layer table; the trainable-parameter count is displayed explicitly instead. |

Note what `trainer.fit` is *not* given: no `val_dataloaders`. There is therefore no validation loop, no early stopping and no learning-rate adaptation — training runs for exactly 10 epochs and stops.

In [ ]:
train_loader = train_dataset.to_dataloader(
    train=True, batch_size=BATCH_SIZE, num_workers=0
)
test_loader = test_dataset.to_dataloader(
    train=False, batch_size=BATCH_SIZE, num_workers=0
)

tft = TemporalFusionTransformer.from_dataset(
    train_dataset,
    hidden_size=64,
    lstm_layers=2,
    attention_head_size=4,
    hidden_continuous_size=32,
    dropout=0.1,
    learning_rate=0.001,
    loss=QuantileLoss(quantiles=list(QUANTILES)),
    output_size=len(QUANTILES),
    optimizer="Adam",
    reduce_on_plateau_patience=0,
    log_interval=-1,
    logging_metrics=[],
)
parameter_count = sum(
    parameter.numel() for parameter in tft.parameters() if parameter.requires_grad
)
display(
    pd.DataFrame(
        [
            {
                "station_id": station_id,
                "device": "mps",
                "trainable_parameters": parameter_count,
            }
        ]
    )
)

trainer = Trainer(
    accelerator="mps",
    devices=1,
    max_epochs=MAX_EPOCHS,
    gradient_clip_val=0.1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)
trainer.fit(tft, train_dataloaders=train_loader)

## Sealed-test quantile forecasts

Predictions are generated only for the filtered sealed-test sequences.

- `mode="quantiles"` returns one value per requested quantile at each horizon, giving a `(n_sequences, 24, 3)` array.
- `return_x=True` also returns the model inputs, which is the only way to recover `decoder_time_idx` — the integer hour each predicted value belongs to.
- `trainer_kwargs` repeats the MPS/no-logging/no-checkpointing setup, because `predict` spins up its own trainer.

Everything after that is verification before a single score is computed: the prediction array must have the expected `(24, 3)` trailing shape, the decoder timestamps must align with it, and the actual values and timestamps looked up by `time_idx` must contain no nulls. Getting this wrong would silently score predictions against the wrong hours, which is exactly the kind of error that produces a plausible number rather than a crash.

The displayed table pins down what was actually scored: the first and last issue time, the first and last target time, the number of forecast origins and the total number of scored values.

In [ ]:
prediction_result = tft.predict(
    test_loader,
    mode="quantiles",
    return_x=True,
    trainer_kwargs={
        "accelerator": "mps",
        "devices": 1,
        "logger": False,
        "enable_checkpointing": False,
        "enable_model_summary": False,
    },
)
quantile_predictions = prediction_result.output.detach().cpu().numpy()
decoder_time_idx = prediction_result.x["decoder_time_idx"].detach().cpu().numpy()
if quantile_predictions.shape[1:] != (PREDICTION_LENGTH, len(QUANTILES)):
    raise ValueError(f"Unexpected TFT prediction shape: {quantile_predictions.shape}")
if decoder_time_idx.shape != quantile_predictions.shape[:2]:
    raise ValueError("TFT decoder timestamps do not align with predictions")

timestamp_lookup = test_data.drop_duplicates("time_idx").set_index("time_idx")[
    "timestamp"
]
actual_lookup = test_data.drop_duplicates("time_idx").set_index("time_idx")[
    "water_level"
]
actual_values = (
    actual_lookup.reindex(decoder_time_idx.ravel())
    .to_numpy()
    .reshape(decoder_time_idx.shape)
)
target_timestamps = (
    timestamp_lookup.reindex(decoder_time_idx.ravel())
    .to_numpy()
    .reshape(decoder_time_idx.shape)
)
issue_timestamps = timestamp_lookup.reindex(decoder_time_idx[:, 0] - 1).to_numpy()
if (
    np.isnan(actual_values).any()
    or pd.isna(target_timestamps).any()
    or pd.isna(issue_timestamps).any()
):
    raise ValueError(
        "Filtered sealed test sequences contain incomplete target timestamps or values"
    )

scored_timestamp_ranges = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "issue_start": issue_timestamps.min(),
            "issue_end": issue_timestamps.max(),
            "target_start": target_timestamps.min(),
            "target_end": target_timestamps.max(),
            "forecast_origins": len(issue_timestamps),
            "scored_values": actual_values.size,
        }
    ]
)
display(scored_timestamp_ranges)

## Accuracy and calibration

A point forecast can only be judged on how far off it is; a quantile forecast also has to be judged on whether its stated uncertainty is honest. Both tables — aggregate, and one row per lead hour — report all of it from the same filtered sealed-test cohort.

**`pinball_loss(actual, prediction, quantile)`** is the loss the model was trained on, reported per quantile. For quantile `q` it charges `q` per unit of under-prediction and `1 - q` per unit of over-prediction, so the p10 forecast is punished heavily for being too high and the p90 for being too low. It is minimised by the true quantile, which is what makes it the correct scoring rule here.

| Metric | What it tells you |
| --- | --- |
| `median_mae`, `median_rmse` | Point accuracy of the p50 forecast, in water-level units, directly comparable with the other stage-4 notebooks. RMSE is dominated by the worst misses; a wide gap from MAE means a few large errors rather than uniformly poor accuracy. |
| `pinball_p10`, `pinball_p50`, `pinball_p90` | Per-quantile loss. Not comparable across quantiles — p10 and p90 losses are naturally smaller than p50 — but comparable across models and horizons at the same quantile. |
| `empirical_80pct_coverage` | Fraction of actual values that fell inside the p10–p90 band. The nominal target is 0.80. Materially below means the model is overconfident; materially above means it is hedging with intervals wider than it needs. |
| `mean_interval_width` | Average p90 minus p10, in water-level units. Coverage is meaningless without it — an infinitely wide interval covers everything and says nothing. Read the two together. |
| `quantile_crossing_rate` | Fraction of predictions where the quantiles come out in the wrong order (p10 above p50, or p50 above p90). Quantile loss does not structurally forbid this, so a non-zero rate is a real diagnostic: it means the model is producing internally inconsistent distributions. |

Reading the per-horizon table matters more here than elsewhere. A well-calibrated forecaster should get *wider* intervals as the horizon grows while holding coverage near 0.80; intervals that stay flat from `t+1` to `t+24` mean the model is not representing how much less it knows about tomorrow.

In [ ]:
def pinball_loss(actual: np.ndarray, prediction: np.ndarray, quantile: float) -> float:
    """Quantile (pinball) loss for one predicted quantile against the actual value."""
    error = actual - prediction
    return float(np.maximum(quantile * error, (quantile - 1.0) * error).mean())


def metric_row(actual: np.ndarray, prediction: np.ndarray) -> dict[str, float]:
    """Compute accuracy and calibration metrics for one actual/prediction slice."""
    p10, p50, p90 = (prediction[..., index] for index in range(len(QUANTILES)))
    crossing = (p10 > p50) | (p50 > p90)
    return {
        "median_mae": float(np.abs(actual - p50).mean()),
        "median_rmse": float(np.sqrt(np.mean((actual - p50) ** 2))),
        "pinball_p10": pinball_loss(actual, p10, 0.10),
        "pinball_p50": pinball_loss(actual, p50, 0.50),
        "pinball_p90": pinball_loss(actual, p90, 0.90),
        "empirical_80pct_coverage": float(((actual >= p10) & (actual <= p90)).mean()),
        "mean_interval_width": float((p90 - p10).mean()),
        "quantile_crossing_rate": float(crossing.mean()),
    }


aggregate_metrics = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "forecast_origins": len(actual_values),
            "scored_values": actual_values.size,
            **metric_row(actual_values, quantile_predictions),
        }
    ]
)
per_horizon_metrics = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "horizon_hours": horizon,
            **metric_row(
                actual_values[:, horizon - 1], quantile_predictions[:, horizon - 1, :]
            ),
        }
        for horizon in range(1, PREDICTION_LENGTH + 1)
    ]
)
display(aggregate_metrics)
display(per_horizon_metrics)

## Forecast inspection and model interpretation

The preview lists the first five forecast origins in long form — one row per (origin, horizon) pair, with the issue timestamp, the target timestamp, the actual value and all three quantiles — so any individual number can be traced back to a specific hour instead of being read off an aggregate.

The interpretation step re-runs prediction with `mode="raw"`, which returns the internal attention and variable-selection tensors rather than just the outputs. `tft.interpret_output(..., reduction="sum")` aggregates them over every sealed-test forecast, and `plot_interpretation` renders them:

- **Encoder and decoder variable importance** — how much weight the variable-selection networks gave each input across the whole test set. This is the one place in stage 4 where a model says which of the 53 Stage-3 predictors it actually used.
- **Attention over the encoder window** — which of the 168 past hours the model attended to, averaged over forecasts. A peak at recent hours is expected; a peak near lag 24 would be evidence of the daily cycle that the SARIMAX diagnostic found to be weak.

Treat these as descriptive, not causal: they show where the fitted model put its weight, not what drives the river. Nothing here is written to disk.

In [ ]:
preview_origins = min(5, len(issue_timestamps))
forecast_preview = pd.DataFrame(
    {
        "issue_timestamp": np.repeat(
            issue_timestamps[:preview_origins], PREDICTION_LENGTH
        ),
        "target_timestamp": target_timestamps[:preview_origins].ravel(),
        "horizon_hours": np.tile(np.arange(1, PREDICTION_LENGTH + 1), preview_origins),
        "actual_water_level": actual_values[:preview_origins].ravel(),
        "p10": quantile_predictions[:preview_origins, :, 0].ravel(),
        "p50": quantile_predictions[:preview_origins, :, 1].ravel(),
        "p90": quantile_predictions[:preview_origins, :, 2].ravel(),
    }
)
display(forecast_preview)

raw_prediction_result = tft.predict(
    test_loader,
    mode="raw",
    return_x=True,
    trainer_kwargs={
        "accelerator": "mps",
        "devices": 1,
        "logger": False,
        "enable_checkpointing": False,
        "enable_model_summary": False,
    },
)
interpretation = tft.interpret_output(raw_prediction_result.output, reduction="sum")
interpretation_figures = tft.plot_interpretation(interpretation)
for name, figure in interpretation_figures.items():
    figure.suptitle(name.replace("_", " "))
    display(figure)